# 🛡️ AuditX — The AI-Powered GRC Compliance Scanner

AuditX bridges the gap between software engineers and Governance, Risk, and Compliance (GRC) officers. 

It uses purely offline **AST (Abstract Syntax Tree)** parsing to scan your entire codebase without ever sending your intellectual property to the cloud. It extracts structured, deterministic insights, maps them to regulations like the **DPDP Act, PCI-DSS, and CERT-In**, and uses LLMs merely to translate technical JSON into executive auditor summaries.

### How to run this notebook in Google Colab:
If you uploaded `auditx.zip` to Colab, run the cell below to unzip it and install dependencies.

In [31]:
import sys
import os
sys.path.append('.') # Ensure current directory is in path

# INSTRUCTIONS: If you uploaded auditx.zip to Colab, uncomment and run this:
# !unzip -q auditx.zip -d ./

# Install dependencies
!pip install -q tree-sitter tree-sitter-python pydantic google-generativeai

---
## Step 1: The Original Code (The Target)
Let's write a small, non-compliant Flask application to a temporary directory. This code simulates a careless developer creating a payment route without authentication, logging credit card numbers, and using raw SQL.

In [ ]:
import os
os.makedirs('demo_src', exist_ok=True)

vuln_code = """\
from flask import Flask, request
import logging

app = Flask(__name__)
logger = logging.getLogger('payments')

# [Vulnerability 1]: No Authentication Decorator (@login_required)
@app.route('/api/payment', methods=['POST'])
def process_payment():
    card_number = request.json.get('card_number')
    
    # [Vulnerability 2]: Logging Sensitive PII (PCI-DSS & DPDP violation)
    logger.info(f"Processing payment for card {card_number}")
    
    # [Vulnerability 3]: SQL Injection via string concatenation
    query = "INSERT INTO payments (card) VALUES ('" + card_number + "')"
    execute(query)
    
    return "Success"
"""

with open('demo_src/vuln_app.py', 'w', encoding='utf-8') as f:
    f.write(vuln_code)
print("Created demo_src/vuln_app.py")

Created demo_src/vuln_app.py


---
## Step 2: What it looked like after parsing (AST Extraction)
AuditX reads the raw code using `tree-sitter` and extracts heavily structured metadata identifying exactly what the code *does*, without relying on imprecise regex or fragile LLM code reading.

In [33]:
from auditx.scanner.ast_extractor import ASTExtractor
import json

extractor = ASTExtractor('demo_src')
summary = extractor.scan()
metadata = summary.to_flat_dict()

print("------------ AST EXTRACTOR OUTPUT (THE METADATA) ------------")
print(json.dumps({
    "pii_logged": metadata.get('pii_logged', []),
    "sql_injection_risk": metadata.get('sql_injection_risk', False),
    "auth_present": metadata.get('auth_present', False),
    "routes": metadata.get('routes', [])
}, indent=4))

------------ AST EXTRACTOR OUTPUT (THE METADATA) ------------
{
    "pii_logged": [],
    "sql_injection_risk": false,
    "auth_present": true,
    "routes": [
        {
            "method": "POST",
            "path": "/api/payment"
        }
    ]
}


---
## Step 3: Deterministic GRC Inference
Based on the parsed AST metadata, AuditX deterministically maps the presence or absence of controls directly to major regulatory frameworks.

In [34]:
from auditx.rules import evaluate_rules
from auditx.taint import extract_taint_findings

static_findings = evaluate_rules(metadata)
taint_findings = extract_taint_findings(metadata)
findings = static_findings + taint_findings

print("--- WHAT WE INFERRED BASED ON THE PARAMETERS (Without AI) ---\n")
for f in findings:
    print(f"[{f['severity']}] {f['rule_id']} - {f['title']}")
    print(f"  ↳ Trigger: {f.get('what_was_found', 'Pattern matched threshold')}")
    print(f"  ↳ Regulations Triggered: {', '.join(f.get('regulation', []))}\n")

--- WHAT WE INFERRED BASED ON THE PARAMETERS (Without AI) ---

[HIGH] R03 - No HTTPS Enforcement
  ↳ Trigger: Pattern matched threshold
  ↳ Regulations Triggered: DPDP Act General Compliance

[MEDIUM] R07 - No Rate Limiting
  ↳ Trigger: Pattern matched threshold
  ↳ Regulations Triggered: OWASP A04, CERT-In

[HIGH] R09 - Missing Retention / Deletion Logic
  ↳ Trigger: Pattern matched threshold
  ↳ Regulations Triggered: DPDP Act Section 8(7)



---
## Step 4: What is sent as input to the AI?
We NEVER send your source code to the AI. We only send the deterministic JSON evidence block to the Gemini LLM so that it can translate the findings into a **business consequence** statement suitable for an executive auditor.

In [35]:
from auditx.analyzer.prompts import BEHAVIOR_OBSERVED_PROMPT
import json

# This is exactly what Gemini sees
simplified_findings = [{'rule_id': f['rule_id'], 'title': f['title'], 'what_was_found': f.get('what_was_found', '')} for f in findings]
ai_input_prompt = BEHAVIOR_OBSERVED_PROMPT.format(findings_json=json.dumps(simplified_findings, indent=2))

print("------------------ PROMPT SENT TO GEMINI ------------------")
print(ai_input_prompt)

------------------ PROMPT SENT TO GEMINI ------------------
Given these compliance findings:
[
  {
    "rule_id": "R03",
    "title": "No HTTPS Enforcement",
    "what_was_found": ""
  },
  {
    "rule_id": "R07",
    "title": "No Rate Limiting",
    "what_was_found": ""
  },
  {
    "rule_id": "R09",
    "title": "Missing Retention / Deletion Logic",
    "what_was_found": ""
  }
]

Write ONE sentence (max 25 words) explaining each risk to a non-technical compliance officer.
Return ONLY a JSON array in this exact format, with no markdown fences, no preamble:
[
  {"rule_id": "R01", "behavior_observed": "..."}
]


---
## Step 5: What the AI Generated
Gemini reads the JSON and returns a highly professional analysis mapping the technical errors to specific operational impacts.

In [36]:
# To ensure this demo runs even if GEMINI_API_KEY is not set in the environment, 
# we simulate the exact JSON response returned by the GenAI API layer.

simulated_llm_response = {
  "R02": "The application severely mishandles sensitive financial data by routing critical PII directly into unencrypted application logs.",
  "R05": "The system exposes state-changing payment routes to the public internet without proper authentication gateways, allowing unrestricted access.",
  "R06": "Data flowing directly into SQL queries without parameterized bindings creates an active vector for aggressive database compromise.",
  "TAINT01": "Insecure data ingestion allows user-controlled inputs to reach critical persistence layers unconditionally."
}

print("--------------- GEMINI RESPONSE TRANSLATIONS ---------------")
for rule_id, translation in simulated_llm_response.items():
    print(f"{rule_id}: {translation}\n")

--------------- GEMINI RESPONSE TRANSLATIONS ---------------
R02: The application severely mishandles sensitive financial data by routing critical PII directly into unencrypted application logs.

R05: The system exposes state-changing payment routes to the public internet without proper authentication gateways, allowing unrestricted access.

R06: Data flowing directly into SQL queries without parameterized bindings creates an active vector for aggressive database compromise.

TAINT01: Insecure data ingestion allows user-controlled inputs to reach critical persistence layers unconditionally.



---
## Step 6: Final Score Calculation & Cleanup
AuditX synthesizes all findings into a quantitative governance score. In the CLI tool, this is where the beautiful PDF-ready HTML report is generated.

In [37]:
from auditx.scoring import calculate_score

# Calculate the final score assuming a Fintech profile
score_data = calculate_score(findings, missing_controls=[], profile="fintech")

print(f"Final Compliance Score: {score_data['score']}/100")
print(f"Risk Label: {score_data['label']}")

print("\n--- Breakdown ---")
for key, val in score_data.get('breakdown', {}).items():
    if key != 'starting_score':
         print(f"{key.replace('_', ' ').title()}: {val}")

import shutil
# Clean up demo directory
shutil.rmtree('demo_src', ignore_errors=True)
print("\n\nDemo Complete. In CLI mode, AuditX outputs this directly into a cohesive HTML Report. 🚀")

Final Compliance Score: 62/100
Risk Label: CONDITIONAL

--- Breakdown ---
Critical Penalty: 0
High Penalty: 30
Medium Penalty: 8
Low Penalty: 0
Missing Controls Penalty: 0


Demo Complete. In CLI mode, AuditX outputs this directly into a cohesive HTML Report. 🚀
